# Phase 10.9 — Ranking Hardening & Failure Analysis

- **Production:** 10.4 baseline in `retrieve_ranked()`
- **LTR:** offline/experimental only
- **Benchmark:** 10 queries / 44 judged relevant IDs (demo scale)

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "productiq").is_dir():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT / "src"))

from productiq.ranking.evaluation import RankingBenchmarkEvaluator, RankingEvaluationConfig
from productiq.ranking.failure_analysis import build_ranking_failure_analysis_report
from productiq.ranking.hardening import RANKING_HARDENING_VERSION, resolve_production_ranking_mode
from productiq.ranking.invariants import validate_ranking_response_invariants
from productiq.retrieval.evaluation.dataset import load_lexical_retrieval_benchmark, default_benchmark_path
from productiq.retrieval.production_ranking_config import ProductionRankingStageConfig

In [ ]:
stage = ProductionRankingStageConfig()
print("hardening_version", RANKING_HARDENING_VERSION)
print("production_ranking_mode", resolve_production_ranking_mode(stage))
print("experimental_ltr_enabled", stage.experimental_ltr.enabled)

In [ ]:
benchmark = load_lexical_retrieval_benchmark(default_benchmark_path(ROOT))
config = RankingEvaluationConfig()
evaluator = RankingBenchmarkEvaluator(config)
# Illustrative list-based evaluation (same population for RRF/baseline lists)
evaluation = evaluator.evaluate_from_lists(
    benchmark,
    rrf_by_query_id={q.query_id: q.relevant_product_ids for q in benchmark.queries},
    baseline_by_query_id={q.query_id: q.relevant_product_ids for q in benchmark.queries},
    pool_by_query_id={q.query_id: q.relevant_product_ids for q in benchmark.queries},
)
report = build_ranking_failure_analysis_report(benchmark, evaluation)
print("failure_records", len(report.records))
if report.records:
    sample = report.records[0]
    print("observed", sample.observed_facts)
    print("hypotheses", sample.diagnosis_hypotheses)

## Limitations

Observed facts come from benchmark artifacts; hypotheses are not causal proof. Do not tune models or claim generalization.